<a href="https://colab.research.google.com/github/michalis0/DataScience_and_MachineLearning/blob/master/Labs/09-unsupervised-learning/2.association-rules.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Lab 9 · Movie recommendations with association rules

## Learning goals

- Turn explicit movie ratings into one basket of liked films per user.
- Mine frequent itemsets with FP-growth and generate directional rules with `mlxtend`.
- Calculate and interpret support, confidence, and lift from both rule output and raw counts.
- Recommend new films from a supplied watch history and explain why each rule matched.

## Before you start

Open the notebook from this lab folder. Run the installation cell below to install `mlxtend` into the notebook kernel. The first run downloads [MovieLens 100K](https://grouplens.org/datasets/movielens/100k/) from GroupLens and caches it in `data/`. To work offline, put the official ZIP at `data/ml-100k.zip` first. Posters are optional.

In [ ]:
%pip install -q mlxtend==0.23.4

## 1. Load the ratings

MovieLens 100K contains 100,000 ratings by 943 users for 1,682 films. The ratings and titles are separate files, so use the movie ID as the key. Keep the IDs through mining and join the titles for display.

In [1]:
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import association_rules, fpgrowth

archive = Path("data/ml-100k.zip")
if not archive.is_file():
    archive.parent.mkdir(parents=True, exist_ok=True)
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    with urlopen(url, timeout=30) as response:
        payload = response.read()
    with ZipFile(BytesIO(payload)) as candidate:
        candidate.getinfo("ml-100k/u.data")
        candidate.getinfo("ml-100k/u.item")
    archive.write_bytes(payload)

with ZipFile(archive) as source:
    with source.open("ml-100k/u.data") as file:
        ratings = pd.read_csv(file, sep="	", names=["user_id", "movie_id", "rating", "timestamp"])
    with source.open("ml-100k/u.item") as file:
        movies = pd.read_csv(file, sep="|", usecols=[0, 1], names=["movie_id", "title"],
                             header=None, encoding="latin-1")

titles = movies.set_index("movie_id")["title"]
assert (len(ratings), ratings.user_id.nunique(), len(movies)) == (100_000, 943, 1_682)
display(ratings.head().join(titles, on="movie_id").drop(columns="timestamp"))

,user_id,movie_id,rating,title
0,196,242,3,Kolya (1996)
1,186,302,3,L.A. Confidential (1997)
2,22,377,1,Heavyweights (1994)
3,244,51,2,Legends of the Fall (1994)
4,166,346,1,Jackie Brown (1997)


## 2. Build a movie basket for each user

Treat a rating of four or five stars as a like. A missing flag means there is no recorded like. It can represent an unrated movie or a lower rating. Keep all users in the denominator, including any with an empty basket.

In [2]:
liked = ratings.loc[ratings.rating.ge(4), ["user_id", "movie_id"]]
basket = (
    pd.crosstab(liked.user_id, liked.movie_id)
    .reindex(index=sorted(ratings.user_id.unique()), fill_value=0)
    .astype(bool)
)
print(f"{len(basket):,} user baskets, {basket.shape[1]:,} films with at least one like")
display(basket.iloc[:5, :6].rename(columns=titles).astype(int))

943 user baskets, 1,447 films with at least one like


movie_id,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995),Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
user_id,,,,,,
1,1,0,1,0,0,1
2,1,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
5,1,0,0,0,0,0


## 3. Mine frequent itemsets and generate rules

Use FP-growth to keep itemsets found in at least 4% of user baskets. Limit the size to three films to keep the walkthrough quick. Keep rules with one recommended movie, at least 35% confidence, and lift above one.

In [3]:
itemsets = fpgrowth(basket, min_support=0.04, use_colnames=True, max_len=3)
rules = association_rules(itemsets, num_itemsets=len(basket), metric="confidence", min_threshold=0.35)
rules = rules.loc[(rules.consequents.map(len) == 1) & rules.lift.gt(1)].copy()
rules["candidate_id"] = rules.consequents.map(lambda items: next(iter(items)))

def name_set(items):
    return " + ".join(titles.loc[sorted(items)])

def rule_table(rows):
    return rows.assign(
        liked=rows.antecedents.map(name_set),
        recommend=rows.candidate_id.map(titles),
    )[["liked", "recommend", "support", "confidence", "lift"]].round(3)

print(f"{len(itemsets):,} frequent itemsets and {len(rules):,} eligible rules")
display(rule_table(rules.sort_values(["lift", "support"], ascending=False).head(6)))

107,555 frequent itemsets and 270,567 eligible rules


,liked,recommend,support,confidence,lift
182334,"Wrong Trousers, The (1993) + Close Shave, A (1...","Grand Day Out, A (1992)",0.047,0.638,11.346
296477,"Terminator, The (1984) + Die Hard: With a Veng...",Die Hard 2 (1990),0.041,0.765,10.016
296471,Terminator 2: Judgment Day (1991) + Die Hard: ...,Die Hard 2 (1990),0.043,0.745,9.763
296470,Terminator 2: Judgment Day (1991) + Die Hard 2...,Die Hard: With a Vengeance (1995),0.043,0.672,9.603
296467,Raiders of the Lost Ark (1981) + Die Hard 2 (1...,Die Hard: With a Vengeance (1995),0.046,0.672,9.600
296475,"Terminator, The (1984) + Die Hard 2 (1990)",Die Hard: With a Vengeance (1995),0.041,0.661,9.445


## 4. Recompute all three metrics by hand

For a rule **A → B**, let **N** be the number of baskets, **n(A)** the number containing all films in A, **n(B)** the number containing B, and **n(A and B)** the number containing both.

| Metric | Calculation | Interpretation |
| --- | --- | --- |
| Support | n(A and B) / N | Share of all users who liked both sides |
| Confidence | n(A and B) / n(A) | Share of users who liked B among users who liked A |
| Lift | confidence / (n(B) / N) | How much more common B is among users who liked A, compared with the overall rate |

Support is symmetric. Confidence is directional. Lift above one indicates positive association in this sample. These co-likes do not demonstrate a causal effect.

In [4]:
example = rules.sort_values(["lift", "support"], ascending=False).iloc[0]
has_a = basket.loc[:, sorted(example.antecedents)].all(axis=1)
has_b = basket[example.candidate_id]
both = int((has_a & has_b).sum())
support = both / len(basket)
confidence = both / int(has_a.sum())
lift = confidence / has_b.mean()

print(f"{name_set(example.antecedents)} → {titles[example.candidate_id]}")
print(f"N={len(basket)}, n(A)={has_a.sum()}, n(B)={has_b.sum()}, n(A and B)={both}")
display(pd.DataFrame({"from counts": [support, confidence, lift],
                      "from mlxtend": [example.support, example.confidence, example.lift]},
                     index=["support", "confidence", "lift"]).round(4))
np.testing.assert_allclose([support, confidence, lift],
                           [example.support, example.confidence, example.lift])

Wrong Trousers, The (1993) + Close Shave, A (1995) → Grand Day Out, A (1992)
N=943, n(A)=69, n(B)=53, n(A and B)=44


,from counts,from mlxtend
support,0.0467,0.0467
confidence,0.6377,0.6377
lift,11.3459,11.3459


## 5. Recommend films for a sample viewer

Match rules whose antecedent is covered by the viewer’s likes. Exclude films already watched, rank by lift, then confidence and support, and keep the best rule per film. This is a recommendation demo. The rankings have not been evaluated on a held-out set.

In [5]:
def recommend(liked_ids, watched_ids=None, k=5):
    liked_ids = set(liked_ids)
    watched_ids = liked_ids | set(() if watched_ids is None else watched_ids)
    matching = rules.loc[
        rules.antecedents.map(lambda antecedent: antecedent.issubset(liked_ids))
        & ~rules.candidate_id.isin(watched_ids)
    ]
    return (
        matching.sort_values(
            ["lift", "confidence", "support", "candidate_id"],
            ascending=[False, False, False, True],
        )
        .drop_duplicates("candidate_id")
        .head(k)
    )

seed_ids = {50, 172}  # Star Wars and The Empire Strikes Back in MovieLens 100K
recommendations = recommend(seed_ids)
print("Viewer likes:", name_set(seed_ids))
display(rule_table(recommendations))
assert len(recommendations) == 5
assert not set(recommendations.candidate_id) & seed_ids

Viewer likes: Star Wars (1977) + Empire Strikes Back, The (1980)


,liked,recommend,support,confidence,lift
75041,"Star Wars (1977) + Empire Strikes Back, The (1...",Star Trek: The Wrath of Khan (1982),0.120,0.412,2.576
3483,"Star Wars (1977) + Empire Strikes Back, The (1...",Indiana Jones and the Last Crusade (1989),0.182,0.628,2.519
216392,"Star Wars (1977) + Empire Strikes Back, The (1...","Lion King, The (1994)",0.104,0.358,2.517
79462,"Star Wars (1977) + Empire Strikes Back, The (1...","Hunt for Red October, The (1990)",0.117,0.401,2.507
70410,"Star Wars (1977) + Empire Strikes Back, The (1...",Jurassic Park (1993),0.125,0.431,2.507


## 6. Optional movie posters

Set the environment variable `TMDB_READ_TOKEN` to your TMDB API read access token before starting Jupyter. The code searches by title and year and only shows a poster for an unambiguous match. MovieLens IDs are separate from TMDB IDs.

This product uses the TMDB API but is not endorsed or certified by TMDB. See the [TMDB image guide](https://developer.themoviedb.org/docs/image-basics).

In [6]:
import json
import os
import re
from urllib.error import URLError
from urllib.parse import urlencode
from urllib.request import Request
from IPython.display import Image

def normalize(text):
    return re.sub(r"[^a-z0-9]", "", text.casefold())

def poster_url(title, token):
    match = re.fullmatch(r"(.+) \((\d{4})\)", title)
    if not match:
        return None
    name, year = match.groups()
    article = re.fullmatch(r"(.+), (The|A|An)", name)
    if article:
        name = f"{article[2]} {article[1]}"
    parameters = urlencode({"query": name, "year": year})
    request = Request(
        "https://api.themoviedb.org/3/search/movie?" + parameters,
        headers={"Authorization": f"Bearer {token}", "Accept": "application/json"},
    )
    with urlopen(request, timeout=10) as response:
        matches = json.load(response)["results"]
    posters = {movie["poster_path"] for movie in matches
               if movie.get("poster_path")
               and movie.get("release_date", "")[:4] == year
               and normalize(name) in {normalize(movie.get("title", "")),
                                       normalize(movie.get("original_title", ""))}}
    if len(posters) == 1:
        return "https://image.tmdb.org/t/p/w185" + posters.pop()
    return None

token = os.environ.get("TMDB_READ_TOKEN")
if token:
    for title in recommendations.candidate_id.map(titles):
        try:
            url = poster_url(title, token)
        except (URLError, TimeoutError):
            print("TMDB request failed. Check network access and your token.")
            break
        if url:
            display(Image(url=url, width=135, alt=title))
        else:
            print(f"No unambiguous poster match for {title}.")
else:
    print("Optional posters skipped. Set TMDB_READ_TOKEN to enable them.")

Optional posters skipped. Set TMDB_READ_TOKEN to enable them.


## Practice

1. Reverse a rule with a single antecedent and compare its support, confidence, and lift.
2. Try a seed profile in a different genre. If no rules match, explain which mining settings could account for that.
3. Rank by confidence first. Compare the recommendations and explain what each ranking favors.
4. Design a test that hides one known like per user. Explain how you would avoid using those hidden likes to mine the rules.

In [7]:
# Write your solution here.